# Week 5 on Colab — train it, read it, score it

This notebook is used on three days:

- **Tuesday:** sections 1 to 4, *the first run*
- **Wednesday:** section 5, *read the run*
- **Thursday and Friday:** sections 6 and 7, *score it*, and again after each data fix

**Runtime → Change runtime type → T4 GPU.**

## 1. Get your code and your data

In [ ]:
import os

REPO_URL = "..."   # the address of YOUR repo

if not os.path.exists("course") and os.path.basename(os.getcwd()) != "course":
    !git clone {REPO_URL} course
if os.path.basename(os.getcwd()) != "course":
    %cd course
!git pull -q
!pip install -q -e . pytest peft bitsandbytes accelerate

## 2. Drive and GPU

Runs are saved on your Drive, so a disconnect costs you minutes.

In [ ]:
import json
from pathlib import Path

import torch
from google.colab import drive

drive.mount("/content/drive")
RUNS = Path("/content/drive/MyDrive/tunelab/week-05")
RUNS.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then start again from the top."
print(torch.cuda.get_device_name(0))
print("runs so far:", sorted(p.name for p in RUNS.iterdir()))

## 3. Three steps on a toy model first

About a minute. It uses your `tokenize_example`, your `collate` and your `to_peft_config` on your real data. If this fails, the real run would have failed too, ten minutes in.

In [ ]:
!python week-05/train.py --smoke --out /content/smoke --force

## 4. The first run

**Before you run this cell, add a row to `logs/run-log.md`:** the run's name, and the one thing that is different from your last run. For the first run: "defaults".

Change `RUN` for every new run. The script refuses to write over a finished one.

If Colab disconnects, run cells 1 and 2 again and then this one, unchanged: it carries on from its last checkpoint.

In [ ]:
RUN = RUNS / "run-01"

!python week-05/train.py --four-bit --out "{RUN}"

## 5. Read the run

Your own `diagnose` and `best_step`, on your own run.

In [ ]:
import importlib.util

import matplotlib.pyplot as plt

spec = importlib.util.spec_from_file_location("diagnose", "week-05/day-3/diagnose.py")
diagnose = importlib.util.module_from_spec(spec)
spec.loader.exec_module(diagnose)

history = json.loads((RUN / "history.json").read_text())
record = json.loads((RUN / "run.json").read_text())

train_rows = [r for r in history if "train" in r]
eval_rows = [r for r in history if "eval" in r]
plt.plot([r["step"] for r in train_rows], [r["train"] for r in train_rows], label="train")
plt.plot([r["step"] for r in eval_rows], [r["eval"] for r in eval_rows], "o-", label="eval")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

print("verdict:", diagnose.diagnose(history), "   best step:", diagnose.best_step(history), "of", record["steps"])
print(f'{record["minutes"]} min, peak GPU {record["peak_gpu_mb"]} MB, '
      f'{record["trainable_parameters"]:,} trainable parameters, dropped {record["dropped"]}')

## 6. Score it

The frozen eval, through the tuned model. Same prompt file, same scorer, same generate function as the baseline.

In [ ]:
!python week-05/predict.py --run "{RUN}"

## 7. Past the single number

By kind, by failure type, what moved, and whether the gain is real.

In [ ]:
from tunelab.analysis import changed, error_table, paired_bootstrap
from tunelab.dataset import load_jsonl
from tunelab.toolcall import score_example, score_predictions

held_out = load_jsonl("data/notes/eval.jsonl")
baseline = load_jsonl("data/notes/baseline_predictions.jsonl")
tuned = load_jsonl("data/notes/tuned_predictions.jsonl")
base_report, tuned_report = score_predictions(held_out, baseline), score_predictions(held_out, tuned)

print(f'{"kind":<11}{"n":>4}{"baseline":>10}{"tuned":>8}')
for kind, row in tuned_report["by_kind"].items():
    print(f'{kind:<11}{row["n"]:>4}{base_report["by_kind"][kind]["accuracy"]:>10.2f}{row["accuracy"]:>8.2f}')
print(f'{"all":<11}{tuned_report["n"]:>4}{base_report["accuracy"]:>10.2f}{tuned_report["accuracy"]:>8.2f}')


def correct(rows):
    by_id = {r["id"]: r["prediction"] for r in rows}
    return [e["id"] in by_id and score_example(e, by_id[e["id"]])["correct"] for e in held_out]


low, high = paired_bootstrap(correct(baseline), correct(tuned))
print(f'\ngain {tuned_report["accuracy"] - base_report["accuracy"]:+.3f}   95% interval {low:+.3f} to {high:+.3f}')

moved = changed(held_out, baseline, tuned)
print({key: len(ids) for key, ids in moved.items()})
print("broken:", moved["broken"])

print("\nfailures of the tuned model, by kind:")
for kind, counts in error_table(held_out, tuned).items():
    print(f"  {kind:<11}", counts)

### The still-wrong examples, in full

Read every one. What do they have in common? Then go and look for that pattern in `train.jsonl`.

In [ ]:
by_id = {r["id"]: r for r in tuned}
for example in held_out:
    if example["id"] in moved["still_wrong"] + moved["broken"]:
        said = example["messages"][-1].get("content", "")
        print(f'\n[{example["id"]}  {example["kind"]}]  {said!r}')
        print("   wanted:", json.dumps(example["expected"], ensure_ascii=False))
        print("   got:   ", repr(by_id[example["id"]]["output"]))

## 8. Keep it

Download the two files and commit them from your laptop. The adapter itself stays on your Drive: you need it again in week 6.

In [ ]:
from google.colab import files

files.download("data/notes/tuned_predictions.jsonl")
files.download("data/notes/tuned.json")